In [ ]:
import pickle
import pandas as pd
import numpy as np
from matminer.datasets import load_dataset
from pymatgen.core.composition import Composition
from matminer.featurizers.composition import ElementProperty, ElementFraction
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

class DataLoader:
    def __init__(self, dataset_name, num_samples):
        self.dataset_name = dataset_name
        self.num_samples = num_samples

    def load_data(self):
        return load_dataset(self.dataset_name)['composition'].to_frame().iloc[:self.num_samples]

    def prepare_compositions(self):
        data = self.load_data()
        data['_Composition'] = data['composition'].apply(self.create_composition)
        return data.dropna().drop_duplicates()

    @staticmethod
    def create_composition(formula):
        try:
            return Composition(formula)
        except ValueError:
            print(f"Error parsing formula: {formula}")
            return None

C:\Users\hp\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\hp\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.5.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Error parsing formula: Eu1.45Pr0.05Ce0.5Sr2Cu2Nb1O10=z
Error parsing formula: Sm1Ba-1Cu3O6.94
Error parsing formula: Y2C2Br0.5!1.5
Error parsing formula: Hg0.3Pb0.7Sr1.75La0.25Cu1O4+2


ElementProperty:   0%|          | 0/9996 [00:00<?, ?it/s]

In [ ]:
class FeatureExtractor:
    def featurize(self, data):
        data = data.dropna()
        ep_featurizer = ElementProperty.from_preset('magpie')
        ep_ftd = ep_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
        
        ef_featurizer = ElementFraction()
        ef_ftd = ef_featurizer.featurize_dataframe(data, col_id='_Composition', ignore_errors=True)
        return ep_ftd, ef_ftd

In [ ]:
class ModelManager:
    def __init__(self):
        self.models = {
            'model_ep': 'sc_ep_rf_cl.pkl',
            'model_efep': 'sc_efep_rf_cl.pkl',
            'model_ef': 'sc_ef_rf_cl.pkl'
        }
        
        for name, path in self.models.items():
            with open(path, 'rb') as file:
                self.models[name] = pickle.load(file)

    def predict(self, model_name, X):
        model = self.models.get(model_name)
        if model:
            return model.predict(X)
        else:
            raise ValueError(f"Model '{model_name}' not found")

    def evaluate(self, y_true, y_pred):
        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "confusion_matrix": confusion_matrix(y_true, y_pred),
            "precision": precision_score(y_true, y_pred),
            "recall": recall_score(y_true, y_pred),
            "f1_score": f1_score(y_true, y_pred),
            "roc_auc_score": roc_auc_score(y_true, y_pred)
        }

In [ ]:
class SuperconductorPredictor:
    def __init__(self, data_loader, feature_extractor, model_manager):
        self.data_loader = data_loader
        self.feature_extractor = feature_extractor
        self.model_manager = model_manager

    def record_indices(self, ep_ftd, ef_ftd, data):
        data["_Composition"] = data['composition'].apply(DataLoader.create_composition).to_frame()
        compositions = data["composition"].tolist()
        print("Composition:", len(compositions))
        present_indices = []
        missing_indices = []
        
        for idx, composition in enumerate(compositions):
            if composition in ep_ftd['composition'].tolist():
                found_index = ep_ftd[ep_ftd['composition'] == composition]["index"].tolist()
                present_indices.extend(found_index)
            else:
                missing_indices.append(idx)
        
        print("PRESENT INDICES", present_indices[-10:])
        print("MISSING INDICES", missing_indices)

        present_ep_ftd = pd.DataFrame()
        present_ef_ftd = pd.DataFrame()
        missing_ep_ftd = pd.DataFrame()
        missing_ef_ftd = pd.DataFrame()

        if present_indices:
            present_ep_ftd = ep_ftd.loc[ep_ftd["index"].isin(present_indices)]
            present_ef_ftd = ef_ftd.loc[ef_ftd["index"].isin(present_indices)]
         
        if missing_indices:
            missing_data = data.iloc[missing_indices]
            missing_ep_ftd, missing_ef_ftd = self.feature_extractor.featurize(missing_data)
        
        final_ep_ftd = pd.concat([present_ep_ftd, missing_ep_ftd], ignore_index=True)
        final_ef_ftd = pd.concat([present_ef_ftd, missing_ef_ftd], ignore_index=True)

        final_nn_ep_ftd = final_ep_ftd.dropna()
        final_nn_ef_ftd = final_ef_ftd.dropna()
        
        final_in_ep_ftd = final_ep_ftd[final_ep_ftd.isnull().any(axis=1)]["composition"].tolist()
        final_in_ef_ftd = final_ef_ftd[final_ef_ftd.isnull().any(axis=1)]["composition"].tolist()
        print(final_in_ep_ftd, "\n", final_in_ef_ftd)
        
        return final_nn_ep_ftd, final_nn_ef_ftd, final_in_ep_ftd, final_in_ef_ftd

    def preprocess_data(self, ep_ftd, ef_ftd):
        def preprocess_for_model1(ep_ftd):
            ep_ftd["having_tc"] = (ep_ftd["Critical Temp"] >= 10).astype(int)
            ep_X = ep_ftd.iloc[:, 4:-1]
            ep_y = ep_ftd['having_tc']
            return ep_X, ep_y

        def preprocess_for_model3(ef_ftd):
            ef_ftd["having_tc"] = (ef_ftd["Critical Temp"] >= 10).astype(int)
            ef_X = ef_ftd.iloc[:, 4:-1]
            ef_y = ef_ftd['having_tc']
            return ef_X, ef_y

        def preprocess_for_model2(ep_ftd, ef_ftd):
            ef_ftd = ef_ftd.iloc[:, 2:]
            ep_ftd = ep_ftd.iloc[:, 2:]
            
            merged_df = pd.merge(ef_ftd, ep_ftd, left_on=["Critical Temp", "_Composition"], right_on=["Critical Temp", "_Composition"], how="inner")
            merged_df["having_tc"] = (merged_df["Critical Temp"] >= 10).astype(int)

            efep_X = merged_df.iloc[:, 2:-1]
            efep_y = merged_df['having_tc']    
            return efep_X, efep_y, merged_df

        ep_X, ep_y = preprocess_for_model1(ep_ftd)
        efep_X, efep_y, _ = preprocess_for_model2(ep_ftd, ef_ftd)
        ef_X, ef_y = preprocess_for_model3(ef_ftd)
        
        return ep_X, ep_y, efep_X, efep_y, ef_X, ef_y

    def ensemble_predict(self, ep_ftd, ef_ftd):
        ep_X, ep_y, efep_X, efep_y, ef_X, ef_y = self.preprocess_data(ep_ftd, ef_ftd)
        
        pred_ep = self.model_manager.predict('model_ep', ep_X)
        pred_efep = self.model_manager.predict('model_efep', efep_X)
        pred_ef = self.model_manager.predict('model_ef', ef_X)
        
        n_samples = len(pred_ep)
        ensemble_pred = np.zeros(n_samples)
        for i in range(n_samples):
            class_counts = np.bincount([pred_ep[i], pred_efep[i], pred_ef[i]])
            ensemble_pred[i] = np.argmax(class_counts)

        ensemble_pred = ensemble_pred.astype(int)
        return ensemble_pred, ep_y, efep_y, ef_y, ef_ftd

    def evaluate_ensemble(self, ensemble_pred, ep_y, efep_y, ef_y):
        eval_results = self.model_manager.evaluate(efep_y, ensemble_pred)
        return eval_results

    def create_result_dataframe(self, ensemble_pred, ef_ftd):
        result_df = pd.DataFrame({
            "composition": ef_ftd["_Composition"],
            "Tc": ef_ftd["Critical Temp"],
            "prediction": ensemble_pred
        })
        return result_df

    def predict_single_element(self, formula):
        composition = Composition(formula)
        single_element_df = pd.DataFrame({'composition': [composition]})
        ep_ftd, ef_ftd = self.feature_extractor.featurize(single_element_df)
        ensemble_pred, _, _, _, _ = self.ensemble_predict(ep_ftd, ef_ftd)
        result_df = self.create_result_dataframe(ensemble_pred, ef_ftd)
        return result_df.iloc[0]

    def predict(self, data):
        if isinstance(data, pd.DataFrame):
            data = data.drop_duplicates()
            ep_ftd, ef_ftd = self.feature_extractor.featurize(data)
            final_nn_ep_ftd, final_nn_ef_ftd, final_in_ep_ftd, final_in_ef_ftd = self.record_indices(ep_ftd, ef_ftd, data)
            ensemble_pred, ep_y, efep_y, ef_y, ef_ftd = self.ensemble_predict(final_nn_ep_ftd, final_nn_ef_ftd)
            result_df = self.create_result_dataframe(ensemble_pred, ef_ftd)
            return result_df
        elif isinstance(data, str):
            return self.predict_single_element(data)
        else:
            raise ValueError("Unsupported data type. Please provide a DataFrame or a chemical formula as a string.")

In [ ]:
# Initialize the components
data_loader = DataLoader("superconductivity2018", 100)
feature_extractor = FeatureExtractor()
model_manager = ModelManager()
predictor = SuperconductorPredictor(data_loader, feature_extractor, model_manager)

# Load and preprocess data for DataFrame prediction
data = data_loader.prepare_compositions()

# Make predictions on the dataframe
result_df = predictor.predict(data)
print("Predictions for DataFrame:")
print(result_df.head())

# Make prediction for a single element
single_element = "Ba2Cu3O7"
single_prediction = predictor.predict(single_element)
print("\nPrediction for Single Element:")
print(single_prediction)